# 04 - Train Damage Segmentation Baseline (6-Channel U-Net EfficientNet-B3)

Kaggle notebook for training multiclass damage segmentation on xBD dataset.
Uses 6-channel (pre+post concatenated) input with EfficientNet-B3 encoder.

In [ ]:
import os
import sys
from pathlib import Path

IS_KAGGLE = os.path.exists('/kaggle/input')

if IS_KAGGLE:
    # Attach xBD dataset + processed_xbd dataset to this notebook
    PROCESSED_DIR = Path('/kaggle/input/processed-xbd/processed_xbd')  # Adjust dataset slug
    OUTPUT_DIR = Path('/kaggle/working')
    sys.path.insert(0, '/kaggle/working')
    !pip install -q segmentation-models-pytorch albumentations
else:
    PROCESSED_DIR = Path('../data/processed/xbd')
    OUTPUT_DIR = Path('../outputs')
    sys.path.insert(0, str(Path('../src').resolve()))

print(f'Processed dir: {PROCESSED_DIR} (exists: {PROCESSED_DIR.exists()})')
print(f'Output dir: {OUTPUT_DIR}')

In [ ]:
import torch
from torch.utils.data import DataLoader
import logging

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')

from rescuevision.data.datasets import XBDDamageSegmentationDataset
from rescuevision.data.transforms import get_dual_segmentation_transforms
from rescuevision.models.unet import create_unet
from rescuevision.models.losses import WeightedCEDiceLoss, compute_class_weights
from rescuevision.training.train_segmentation import train_segmentation
from rescuevision.utils.seed import set_seed

## Configuration

In [ ]:
config = {
    "seed": 42,
    "device": "cuda",
    "dataset": {
        "processed_dir": str(PROCESSED_DIR),
        "split_csv": str(PROCESSED_DIR / "metadata" / "splits_event_aware.csv"),
        "image_size": 512,
        "batch_size": 4,
        "num_workers": 2,
    },
    "model": {
        "encoder": "efficientnet-b3",
        "encoder_weights": "imagenet",
        "in_channels": 6,
        "classes": 5,
    },
    "training": {
        "epochs": 60,
        "lr": 5e-5,
        "weight_decay": 1e-5,
        "scheduler": "cosine",
        "mixed_precision": True,
        "early_stopping_patience": 10,
        "save_best_metric": "miou",
        "gradient_accumulation_steps": 2,
    },
    "loss": {
        "ce_weight": 0.5,
        "dice_weight": 0.5,
        "ignore_index": 255,
    },
    "augmentation": {
        "horizontal_flip": 0.5,
        "vertical_flip": 0.5,
        "rotate90": 0.5,
        "brightness_contrast": 0.2,
        "normalize": True,
    },
}

set_seed(config["seed"])
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name()}')

## Create Datasets

In [ ]:
train_transform = get_dual_segmentation_transforms(
    config["augmentation"], phase="train", image_size=512,
)
val_transform = get_dual_segmentation_transforms(
    config["augmentation"], phase="val", image_size=512,
)

train_dataset = XBDDamageSegmentationDataset(
    processed_dir=config["dataset"]["processed_dir"],
    split_csv=config["dataset"]["split_csv"],
    split="train",
    transform=train_transform,
    image_size=512,
)

val_dataset = XBDDamageSegmentationDataset(
    processed_dir=config["dataset"]["processed_dir"],
    split_csv=config["dataset"]["split_csv"],
    split="val",
    transform=val_transform,
    image_size=512,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=config["dataset"]["batch_size"],
    shuffle=True,
    num_workers=2,
    pin_memory=True,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=config["dataset"]["batch_size"],
    shuffle=False,
    num_workers=2,
    pin_memory=True,
)

print(f'Train: {len(train_dataset)} tiles, Val: {len(val_dataset)} tiles')

## Compute Class Weights

In [ ]:
import numpy as np

# Sample class frequencies from training masks
sample_labels = []
for i in range(min(len(train_dataset), 2000)):
    sample = train_dataset[i]
    mask_np = sample["mask"].numpy()
    valid = mask_np != 255
    sample_labels.extend(mask_np[valid].flatten().tolist())

class_weights = compute_class_weights(sample_labels, num_classes=5)
class_weights = class_weights.to(device)
print(f'Class weights: {class_weights}')

## Create Model and Train

In [ ]:
model = create_unet(
    encoder_name=config["model"]["encoder"],
    encoder_weights=None,  # Will adapt first conv for 6 channels
    in_channels=config["model"]["in_channels"],
    classes=config["model"]["classes"],
)

# Adapt first conv layer to accept 6 channels from pretrained 3-channel weights
def adapt_first_conv(model, in_channels=6):
    """Adapt encoder first conv to accept more input channels.
    Copies pretrained weights by averaging across the new channel dimension.
    """
    encoder = model.encoder
    # Load pretrained 3-channel weights first
    import segmentation_models_pytorch as smp
    pretrained_unet = smp.Unet(
        encoder_name='efficientnet-b3',
        encoder_weights='imagenet',
        in_channels=3,
        classes=5,
    )
    # Get the first conv from pretrained
    pretrained_conv = pretrained_unet.encoder.conv_stem
    # Get current first conv
    current_conv = encoder.conv_stem
    
    new_weight = current_conv.weight.data.clone()
    # Average pretrained weights across input channels and replicate
    pretrained_w = pretrained_conv.weight.data  # (out, 3, k, k)
    avg_weight = pretrained_w.mean(dim=1, keepdim=True)  # (out, 1, k, k)
    new_weight[:, :3] = pretrained_w  # First 3 channels get pretrained weights
    new_weight[:, 3:] = avg_weight.expand_as(new_weight[:, 3:])  # Rest get averaged
    current_conv.weight.data = new_weight
    print(f'Adapted first conv from 3 to {in_channels} channels')

adapt_first_conv(model, in_channels=6)

loss_fn = WeightedCEDiceLoss(
    class_weights=class_weights,
    ce_weight=config["loss"]["ce_weight"],
    dice_weight=config["loss"]["dice_weight"],
    ignore_index=config["loss"]["ignore_index"],
)

output_dir = OUTPUT_DIR / "damage_seg_effnetb3"

best_metrics = train_segmentation(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    loss_fn=loss_fn,
    config=config,
    output_dir=str(output_dir),
    is_binary=False,
    num_classes=5,
    ignore_index=255,
)

print(f'\nBest metrics: {best_metrics}')

## Visualize Predictions

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

from rescuevision.constants import DAMAGE_COLOR_MAP

# Create damage colormap
damage_colors = [DAMAGE_COLOR_MAP[i] for i in range(5)]
damage_cmap = mcolors.ListedColormap(
    [[c[0]/255, c[1]/255, c[2]/255] for c in damage_colors]
)
damage_norm = mcolors.BoundaryNorm(range(6), damage_cmap.N)
damage_names = ['background', 'no-damage', 'minor', 'major', 'destroyed']

model.eval()
model.to(device)

fig, axes = plt.subplots(3, 4, figsize=(20, 15))

for i in range(4):
    batch = next(iter(val_loader))
    images = batch["image"][:1].to(device)  # (1, 6, H, W)
    masks_gt = batch["mask"][:1].numpy()
    
    with torch.no_grad():
        pred = model(images).argmax(dim=1).squeeze().cpu().numpy()
    
    # Display pre-disaster image (first 3 channels)
    pre_img = images[0, :3].cpu().numpy().transpose(1, 2, 0)
    pre_img = (pre_img * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406]))
    pre_img = np.clip(pre_img, 0, 1)
    
    axes[0, i].imshow(pre_img)
    axes[0, i].set_title(f'Tile: {batch["tile_id"][0][:30]}')
    axes[1, i].imshow(masks_gt[0], cmap=damage_cmap, norm=damage_norm)
    axes[1, i].set_title('Ground Truth')
    axes[2, i].imshow(pred, cmap=damage_cmap, norm=damage_norm)
    axes[2, i].set_title('Prediction')
    
    for ax in axes[:, i]:
        ax.axis('off')

# Add legend
import matplotlib.patches as mpatches
legend_patches = [mpatches.Patch(color=damage_cmap(i), label=damage_names[i]) for i in range(5)]
fig.legend(handles=legend_patches, loc='lower center', ncol=5, fontsize=12)

plt.tight_layout()
plt.savefig(output_dir / 'predictions.png', dpi=150, bbox_inches='tight')
plt.show()

## Save Confusion Matrix

In [ ]:
from rescuevision.evaluation.confusion import compute_confusion_matrix, plot_confusion_matrix

all_preds = []
all_targets = []

model.eval()
with torch.no_grad():
    for batch in val_loader:
        images = batch["image"].to(device)
        masks = batch["mask"].numpy()
        preds = model(images).argmax(dim=1).cpu().numpy()
        
        all_preds.append(preds.flatten())
        all_targets.append(masks.flatten())

all_preds = np.concatenate(all_preds)
all_targets = np.concatenate(all_targets)

cm = compute_confusion_matrix(all_preds, all_targets, num_classes=5, ignore_index=255)
plot_confusion_matrix(
    cm,
    class_names=damage_names,
    title='Damage Segmentation Confusion Matrix',
    save_path=output_dir / 'confusion_matrix.png',
)
print('Confusion matrix saved.')

## Save Checkpoint

In [ ]:
if IS_KAGGLE:
    print('Checkpoint saved at:', output_dir / 'checkpoints')
    print('Save this output as a Kaggle dataset for the inference notebook.')
    !ls -la {output_dir}/checkpoints/
else:
    print('Training complete. Checkpoint saved.')